# eph_08 — Waveform-axis projection of RT encoding

Reproduces the poster figure `rt_response_projection_abs`: per-unit RT encoding
strength `|T_rt|` (response window) vs each unit's **projection onto the waveform
CCA axis** — the cell-type / waveform structural axis from the upstream pipeline.

This is a near-verbatim, trimmed port of the wf_axis path in
`spatial_axis_comparison_rt_encoding_update.ipynb` (which adapted upstream code).
It is intentionally minimal — just enough to load the inputs and produce the
mapping figure.

**Code Ocean only.** It needs `combined_unit_tbl.pkl`, per-session spike-summary
pickles, the LC CCF mesh, and the waveform-feature CSV — none of which are
available locally. Run locally it just prints skip messages and exits clean.

## 1. Setup

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from plotstyle import apply_style, save_fig
apply_style()

if Path("/root/capsule").exists():
    ENV     = "codeocean"
    SCRATCH = Path("/root/capsule/scratch")
    DATA    = Path("/root/capsule/data")
else:
    ENV     = "local"
    SCRATCH = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local").parent
    DATA    = SCRATCH

IS_CO    = ENV == "codeocean"
FIG_DIR  = SCRATCH / "figures" / "eph_08_waveform_axis"
SAVE_FIG = False
N_BOOT   = 2000
print(f"ENV={ENV}")
if not IS_CO:
    print("eph_08 is Code Ocean only (needs combined_unit_tbl, spike summaries, "
          "LC mesh, waveform CSV). Cells below will skip locally.")

## 2. Load units, compute RT encoding, merge CCF coordinates

Per-unit OLS `spike_count ~ 1 + log(RT)` in the response window, merged onto the
filtered unit table that carries CCF coordinates (`x_ccf`, `y_ccf`, `z_ccf`).

In [ ]:
if not IS_CO:
    print("[skip] unit loading + RT encoding (Code Ocean only)")
else:
    import os, re, pickle, json as _json
    import statsmodels.api as sm
    from statsmodels.stats.multitest import multipletests
    from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_analysis import (
        get_session_name_from_path, session_already_done,
    )
    from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import get_session_prefix

    # --- combined unit table + tongue-QC session filter ---
    with open(SCRATCH / "combined_unit_tbl.pkl", "rb") as f:
        combined_ephys_data = pickle.load(f)

    base_dirs = [SCRATCH / "session_analysis_mlk"]
    COVERAGE_MIN, DURATION50_MIN = 90.0, 0.06
    rows_pass = []
    for base_dir in base_dirs:
        if not base_dir.exists():
            continue
        for subdir in base_dir.iterdir():
            if not (subdir.is_dir() and session_already_done(subdir)):
                continue
            try:
                with open(subdir / "tongue_quality_stats.json") as f:
                    d = _json.load(f)
            except Exception:
                continue
            cov = float(d.get("coverage_pct", 0.0))
            dur50 = float(d.get("percentiles", {}).get("duration", {}).get("0.5", 0.0))
            if cov > COVERAGE_MIN and dur50 > DURATION50_MIN:
                rows_pass.append(subdir)
    session_prefix_allow = {
        get_session_prefix(get_session_name_from_path(str(p))) for p in rows_pass
    }
    print(f"Sessions passing tongue QC: {len(rows_pass)}")

    # --- unit quality criteria ---
    DEFAULT_CRITERIA = {
        "isi_violations": {"bounds": [0.0, 0.1]}, "p_max": {"bounds": [0.5, 1.0]},
        "lat_max_p": {"bounds": [0.005, 0.02]}, "eu": {"bounds": [0.0, 0.25]},
        "corr": {"bounds": [0.95, 1.0]}, "qc_pass": {"items": [True]},
        "peak": {"bounds": [-1000, 0]}, "trial_count": {"bounds": [100, 2000]},
        "in_df": {"items": [True]},
    }
    def filter_by_criteria(df, criteria):
        mask = pd.Series(True, index=df.index)
        for col, rule in criteria.items():
            if "bounds" in rule:
                lo, hi = rule["bounds"]
                mask &= df[col].between(lo, hi, inclusive="both")
            if "items" in rule:
                mask &= df[col].isin(rule["items"])
        return df.loc[mask].copy()

    combined_ephys_data = combined_ephys_data.copy()
    combined_ephys_data["session_prefix"] = combined_ephys_data["session"].map(get_session_prefix)
    filtered_ephys = filter_by_criteria(combined_ephys_data, DEFAULT_CRITERIA)
    filtered_ephys = filtered_ephys.loc[
        filtered_ephys["session_prefix"].isin(session_prefix_allow)
    ].copy()
    print(f"Units after QC + session filter: {len(filtered_ephys)}")

### Spike-count windows — read before running

The counts below use a **500 ms post-cue response window and a 2 s pre-cue baseline**.
These are the windows of the run that produced the reference figure: `code/archive/
spatial_axis_comparison_rt_encoding.ipynb` cell 9 (`execution_count` 7) builds
`all_counts_df` with `count_window_s=(0.0, 0.5)`, `baseline_window_s=(-2, 0.0)`, and the
figure is written at `execution_count` 37 of the same run to
`/root/capsule/scratch/figures/poster/rt_response_projection_abs.svg` (2026-05-04,
committed 16 min later in `2f20188`). They also match the published analysis, which uses a
500 ms window after the go cue and a 2 s pre-cue baseline.

They are **not** the `eph_01`-`eph_06` windows (0-200 ms response, -1-0 s baseline).
Earlier drafts of this notebook used those, inherited from a commented-out `cfg` in
`spatial_axis_comparison_rt_encoding_update.ipynb` — a file created *after* the figure, in
the same commit, whose commented block had already been changed. With 0-200 ms / -1-0 s the
waveform-axis projection gives Spearman r = 0.013; the reference reports r = 0.184.

**Check after running:** the response window should give **50 nominally significant /
43 FDR significant** units out of 103 (baseline: 44 / 38). If it reports 42 / 36, the
0-200 ms / -1-0 s windows are still in effect.

In [ ]:
if not IS_CO:
    print("[skip] RT encoding stats + CCF merge (Code Ocean only)")
else:
    # Build the trial x unit spike-count table in-notebook (there is no cached
    # all_counts_df on Code Ocean -- /root/capsule/scratch is not persistent).
    # Windows are the reference run's, NOT eph_01-eph_06's -- see the note above.
    from data_loading import load_units_with_spike_times
    from ephys_utils import AnalysisConfig, build_all_counts_df

    cfg = AnalysisConfig(
        align_key="goCue",
        count_window_s=(0.0, 0.5),      # 500 ms post-cue response window
        baseline_window_s=(-2.0, 0.0),  # 2 s pre-cue baseline
        min_trials_per_group=20,
    )
    units_with_spikes = load_units_with_spike_times(
        filtered_ephys, str(DATA / "LC-NE_scratch_data_1")
    )
    all_counts_df = build_all_counts_df(units_with_spikes, cfg, base_dirs)
    print("all_counts_df:", all_counts_df.shape)

    def build_rt_encoding_stats(df, *, rt_col="reaction_time_firstmove",
                                count_col="spike_count", session_col="session",
                                unit_col="unit_id", min_trials=50, alpha=0.05):
        """OLS spike_count ~ 1 + z(log RT); per (session_prefix, unit) T/p/coef + FDR."""
        def canon_unit(x):
            try: return str(int(float(x)))
            except Exception: return str(x)
        c = df.copy()
        c[session_col] = c[session_col].astype(str)
        c["unit"] = c[unit_col].map(canon_unit)
        c["session_prefix"] = c[session_col].map(get_session_prefix)

        rows = []
        for (sp, u), g in c.groupby(["session_prefix", "unit"]):
            g = g.dropna(subset=[rt_col, count_col])
            rt = g[rt_col].to_numpy(float); y = g[count_col].to_numpy(float)
            m = np.isfinite(rt) & (rt > 0) & np.isfinite(y)
            rt, y = rt[m], y[m]
            n = int(rt.size)
            rec = {"session_prefix": sp, "unit": u, "T_rt": np.nan, "p_rt": np.nan,
                   "coef_rt": np.nan, "n_trials": n}
            if n >= min_trials and np.std(rt) > 0 and np.std(y) > 0:
                x = np.log(rt); x = (x - x.mean()) / x.std()
                try:
                    res = sm.OLS(y, sm.add_constant(x)).fit()
                    rec.update(T_rt=float(res.tvalues[1]), p_rt=float(res.pvalues[1]),
                               coef_rt=float(res.params[1]))
                except Exception:
                    pass
            rows.append(rec)
        out = pd.DataFrame(rows)
        out["q_rt"] = np.nan
        mm = out["p_rt"].notna()
        if mm.any():
            _, q, _, _ = multipletests(out.loc[mm, "p_rt"].values, alpha=alpha, method="fdr_bh")
            out.loc[mm, "q_rt"] = q
        return out

    rt_stats = build_rt_encoding_stats(all_counts_df)
    n_nom = int((rt_stats["p_rt"] < 0.05).sum())
    n_fdr = int((rt_stats["q_rt"].fillna(1) < 0.05).sum())
    print(f"RT encoding stats for {len(rt_stats)} units; "
          f"{n_nom} nominally sig, {n_fdr} FDR sig")
    print("  reference (500 ms / 2 s windows): 50 nominally sig, 43 FDR sig")

    # Merge onto filtered_ephys (carries x_ccf/y_ccf/z_ccf)
    def canon_unit(x):
        try: return str(int(float(x)))
        except Exception: return str(x)
    features_combined = filtered_ephys.copy()
    features_combined["session_prefix"] = features_combined["session"].map(get_session_prefix)
    features_combined["unit_str"] = features_combined["unit"].map(canon_unit)
    rt_stats["unit_str"] = rt_stats["unit"].astype(str)
    features_combined = features_combined.merge(
        rt_stats, on=["session_prefix", "unit_str"], how="left", suffixes=("", "_rt"),
    )
    both = (features_combined["T_rt"].notna()
            & features_combined[["x_ccf", "y_ccf", "z_ccf"]].notna().all(axis=1)).sum()
    print(f"Units with RT encoding + CCF coords: {both}")

### Anatomical filter: `z_ccf` bounds

The reference run excludes units whose CCF dorsoventral coordinate falls outside the LC
range `z_ccf` ∈ [-5.2, -3.5] (LPS mm). On the current unit table this drops exactly one
unit — `behavior_758017_2025-02-06_11-26-14` unit 85, `z_ccf = -2.174`, which sits 2.25 mm
from the LC mesh centroid while every other unit is within 0.79 mm — taking the projection
from n = 100 to the reference's n = 99.

Unit QC in `data_loading` filters on spike-sorting quality only (`isi_violations`, `p_max`,
`snr`, `qc_pass`, ...) and never checks anatomy, so this is the only step that removes
mislocalised units.

The cell is destructive on `features_combined`: re-running it after the filter has applied
reports 0 units outside, which is expected, not a failure. (The reference notebook's stored
output shows exactly that — it reports 0 outside yet n falls from 100 to 99, because the
cell had already been run once.)

In [ ]:
if not IS_CO:
    print("[skip] z_ccf anatomical filter (Code Ocean only)")
else:
    Z_CCF_BOUNDS = (-5.2, -3.5)   # LC dorsoventral range, bregma-relative LPS mm

    outside = (features_combined["z_ccf"].notna()
               & ~features_combined["z_ccf"].between(*Z_CCF_BOUNDS))
    print(f"z_ccf outside {Z_CCF_BOUNDS}: {int(outside.sum())} unit(s)")
    if outside.any():
        print(features_combined.loc[outside, ["session", "unit", "z_ccf"]]
              .to_string(index=False))

    features_combined = features_combined.loc[
        features_combined["z_ccf"].isna()
        | features_combined["z_ccf"].between(*Z_CCF_BOUNDS)
    ].copy()

    both = (features_combined["T_rt"].notna()
            & features_combined[["x_ccf", "y_ccf", "z_ccf"]].notna().all(axis=1)).sum()
    print(f"Units with T_rt + CCF coords after filter: {both}   (reference: 99)")

## 3. CCF coordinate setup + LC mesh centroid

Coordinates are bregma-centered LPS mm with **ML folded to the positive (right) side**,
so units recorded on either hemisphere contribute to one gradient.

The `+ML` direction is not arbitrary: the structural axes are *fitted* in `+ML` space
(`ccf_wf[:, ml] = np.abs(...)`), and the LC mesh itself lies entirely at positive ML
(0.519–1.311 mm, 0 of 40208 vertices negative). Folding units to `-ML` — as earlier drafts
of this notebook did — flips the ML component of every projection relative to the axis it is
projected onto. On the current unit table that shifts the waveform-axis Spearman r by about
+0.01 (reference convention 0.184; `-ML` gives 0.19).

The mesh centroid (also `+ML`) centers coordinates before projecting. Centering is a pure
translation along the axis, so it shifts the plotted x-axis but leaves the correlation
unchanged; this notebook keeps the `20250418_transformed_remesh_10_ccf25.obj` mesh rather
than the reference's `new_core_mesh.obj`, so the x-axis is offset from the reference figure
by ~0.19 mm while r, p and n match.

In [ ]:
if not IS_CO:
    print("[skip] CCF setup + mesh (Code Ocean only)")
else:
    from trimesh import load_mesh
    from ccf_utils import pir_to_lps

    ml, ap, dv = 0, 1, 2
    BREGMA_LPS_MM  = np.array([-5.7, 5.4, -0.45], dtype=float)
    BREGMA_PIR_VOX = np.array([216, 18, 228], dtype=float)
    CCF_RES_UM = 25.0

    def ccf_points_lps_mm(df, fold_right=True):
        """Nx3 bregma-centered LPS mm coords; fold ML onto the positive side.

        Positive ML matches the space the structural axes are fitted in and the
        side the LC mesh occupies -- see the note above.
        """
        ccfs = df[["x_ccf", "y_ccf", "z_ccf"]].to_numpy(dtype=float) - BREGMA_LPS_MM
        if fold_right:
            ccfs[:, ml] = np.abs(ccfs[:, ml])
        return ccfs

    MESH_PATH = (DATA / "LC-NE_scratch_data_1" / "combined" / "ccf_maps"
                 / "20250418_transformed_remesh_10_ccf25.obj")
    mesh = load_mesh(str(MESH_PATH))
    mesh_verts_pir_vox = np.array(mesh.vertices, dtype=float)
    mesh_verts_pir_mm  = (mesh_verts_pir_vox - BREGMA_PIR_VOX) * (CCF_RES_UM / 1000.0)
    mesh_verts_lps_mm  = pir_to_lps(mesh_verts_pir_mm)
    mesh_centroid = np.mean(mesh_verts_lps_mm, axis=0)
    mesh_centroid[ml] = np.abs(mesh_centroid[ml])   # +ML, matching the unit coords
    print(f"Mesh centroid (LPS mm, +ML): {mesh_centroid}")

## 4. Fit the waveform CCA axis (upstream waveform features)

The waveform axis is the 3D direction maximally correlated with waveform-feature
variation across CCF space (canonical correlation analysis, bootstrapped).

The fit comes from `spatial_axes.bootstrap_spatial_axis_cca` — the same
implementation `eph_09` uses, so the two notebooks cannot drift apart. This
notebook previously carried a private inline copy of it.

In [ ]:
if not IS_CO:
    print("[skip] waveform CCA axis (Code Ocean only)")
else:
    from spatial_axes import bootstrap_spatial_axis_cca

    # The waveform-feature CSV is a SEPARATELY ATTACHED data asset, not part of
    # LC-NE_scratch_data_1 -- the archived older version of this analysis hardcoded
    # "results-59472bbb-4c3a-40f9-a1f5-b0c5113e4ab9-waveforms_np". The `_update` source
    # rewrote that to FIG_PREP_DIR/waveforms_np, which is the upstream layout, not this capsule's.
    # Try both, then glob, so a re-attached asset with a different id still resolves.
    def find_wf_features_csv(data_root):
        """Locate the waveform combined_features.csv across known/likely mount points."""
        cands = [
            data_root / "LC-NE_scratch_data_1" / "combined" / "waveforms_np" / "combined_features.csv",
            data_root / "results-59472bbb-4c3a-40f9-a1f5-b0c5113e4ab9-waveforms_np" / "combined_features.csv",
        ]
        cands += sorted(data_root.glob("*waveforms_np*/combined_features.csv"))
        cands += sorted(data_root.glob("**/combined_features.csv"))
        seen = set()
        for c in cands:
            if c in seen:
                continue
            seen.add(c)
            if c.exists():
                return c
        raise FileNotFoundError(
            "combined_features.csv not found under %s. Tried:\n  %s"
            % (data_root, "\n  ".join(str(c) for c in seen))
        )

    FIG_PREP_DIR = DATA / "LC-NE_scratch_data_1" / "combined"
    with open(FIG_PREP_DIR / "combine_unit_tbl" / "combined_unit_tbl.pkl", "rb") as f:
        han_units = pickle.load(f)
    wf_csv = find_wf_features_csv(DATA)
    print(f"waveform features: {wf_csv}")
    wf_feats = pd.read_csv(wf_csv)
    wf_feats = wf_feats.merge(
        han_units[["session", "unit", "x_ccf", "y_ccf", "z_ccf"]],
        on=["session", "unit"], how="left",
    )
    wf_feature_cols = [
        "post_w", "trough_post_ratio_1D", "post_trough_slope", "pre_slope",
        "symmetry_slope_div_log", "symmetry_trough_dis", "symmetry_inte_div_log",
    ]
    ccf_wf = wf_feats[["x_ccf", "y_ccf", "z_ccf"]].values - BREGMA_LPS_MM
    ccf_wf[:, ml] = np.abs(ccf_wf[:, ml])
    valid_wf = (~np.any(np.isnan(ccf_wf), axis=1)
                & ~np.any(np.isnan(wf_feats[wf_feature_cols].values), axis=1))
    res_wf = bootstrap_spatial_axis_cca(
        wf_feats[wf_feature_cols].values[valid_wf], ccf_wf[valid_wf],
        n_boot=N_BOOT, seed=16, align_to_observed=True,
    )
    axis_wf = res_wf["axis_unit"]
    print(f"Waveform axis: {axis_wf}  (n={int(valid_wf.sum())})")

## 5. Projection scatter: `|T_rt|` (response) vs waveform-axis projection

The poster `rt_response_projection_abs` figure: each unit's absolute RT-encoding
T-stat against its projection onto the waveform axis, with an OLS fit + 95% CI
band and a Spearman correlation.

In [ ]:
if not IS_CO:
    print("[skip] projection figure (Code Ocean only)")
else:
    from scipy.stats import spearmanr

    def get_regression_CI(x, y, n_pts=100, ci=0.95):
        from scipy import stats as sp_stats
        res = sm.OLS(y, sm.add_constant(x)).fit()
        x_fit = np.linspace(x.min(), x.max(), n_pts)
        X_fit = sm.add_constant(x_fit)
        y_fit = res.predict(X_fit)
        y_var = np.array([xi @ res.cov_params() @ xi for xi in X_fit])
        se = np.sqrt(y_var)
        t_crit = sp_stats.t.ppf((1 + ci) / 2, res.df_resid)
        return y_fit, x_fit, y_fit - t_crit * se, y_fit + t_crit * se

    mask = (features_combined["T_rt"].notna()
            & features_combined[["x_ccf", "y_ccf", "z_ccf"]].notna().all(axis=1))
    fc = features_combined.loc[mask]
    coords = ccf_points_lps_mm(fc, fold_right=True) - mesh_centroid
    proj = coords @ axis_wf
    feat = np.abs(fc["T_rt"].to_numpy(dtype=float))  # |T_rt|, response (abs)

    ok = np.isfinite(proj) & np.isfinite(feat)
    proj_v, feat_v = proj[ok], feat[ok]

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(proj_v, feat_v, c=feat_v, cmap="viridis", s=25, alpha=0.7,
               edgecolor="white", linewidth=0.3)
    if len(proj_v) > 4:
        r, p = spearmanr(proj_v, feat_v)
        y_fit, x_fit, lo, hi = get_regression_CI(proj_v, feat_v)
        ax.plot(x_fit, y_fit, color="black", linewidth=1.5)
        ax.fill_between(x_fit, lo, hi, color="black", alpha=0.15)
        ax.set_title(f"|T_rt| (response) vs wf_axis\nr={r:.2f}, p={p:.3g}, n={len(proj_v)}",
                     color=("red" if p < 0.05 else "black"), fontsize=11)
    ax.axhline(0, ls=":", color="gray", lw=0.5)
    ax.axvline(0, ls=":", color="gray", lw=0.5)
    ax.set_xlabel("Projection on wf_axis")
    ax.set_ylabel("|T_rt| (response, abs)")
    plt.tight_layout()
    save_fig(fig, "rt_response_projection_abs", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()